In [ ]:
# -*- coding: utf-8 -*-
"""
第一部分：标准EMD二分类数据预处理（无任何阈值筛选，全保留）
"""
import os
os.environ['PYTHONHASHSEED'] = '42'
import random
import numpy as np
random.seed(2024)
np.random.seed(2024)

import pandas as pd
from PyEMD import EMD
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

class StandardEMDProcessor:
    def __init__(self, n_imfs=8):
        self.n_imfs = n_imfs
        self.emd = EMD()

    def _standard_denoise(self, sample):
        orig_signal = sample.copy()
        try:
            imfs = self.emd(orig_signal, max_imf=self.n_imfs)
            residue = orig_signal - np.sum(imfs, axis=0)
            if len(imfs) < 2: 
                return orig_signal
            
            # 标准EMD：不筛选，全部保留，直接相加
            reconstructed = np.sum(imfs, axis=0) + residue
            
            # 幅值与长度校准
            if np.max(reconstructed) != 0 and np.max(orig_signal) != 0:
                reconstructed = reconstructed * (np.max(orig_signal) / np.max(reconstructed))
            if len(reconstructed) > len(orig_signal):
                reconstructed = reconstructed[:len(orig_signal)]
                
            return reconstructed
        except Exception:
            return orig_signal

    def process_data(self, X_data):
        print(f"=== 标准 EMD 处理 {len(X_data)} 个样本 ===")
        processed_signals = []
        for sample_idx, sample in enumerate(X_data):
            if sample_idx % 100 == 0:
                print(f"处理进度: {sample_idx}/{len(X_data)}")
            processed_signals.append(self._standard_denoise(sample))
        return np.array(processed_signals)

def main():
    print("=== 第一阶段：标准 EMD 分解预处理 ===")
    
    # ⚠️ 请确保路径与你的主模型完全一致
    file_path = r'E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\xin\Electricity-Theft-Detection--master\sample.xlsx'
    
    try:
        df = pd.read_excel(file_path)
    except Exception:
        try:
            df = pd.read_csv(file_path, encoding='utf-8')
        except Exception:
            df = pd.read_csv(file_path, encoding='ANSI')

    if 'Unnamed: 0' in df.columns:
        df.drop(['Unnamed: 0'], axis=1, inplace=True)
    
    Y = df['output'].values
    X = df.drop(['output'], axis=1).values
    
    print(f"原始标签分布，正常样本(0): {np.sum(Y==0)}, 窃电样本(1): {np.sum(Y==1)}")
    
    # ⚠️ 必须与主模型、固定阈值模型完全一致的切分比例 (test_size=0.2 还是 0.4，请统一)
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, Y, test_size=0.2, random_state=42, stratify=Y
    )
    
    processor = StandardEMDProcessor(n_imfs=8)
    X_train = processor.process_data(X_train_raw)
    X_test = processor.process_data(X_test_raw)
    
    output_dir = "standard_emd_data"
    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, "X_train.npy"), X_train)
    np.save(os.path.join(output_dir, "y_train.npy"), y_train_raw)
    np.save(os.path.join(output_dir, "X_test.npy"), X_test)
    np.save(os.path.join(output_dir, "y_test.npy"), y_test_raw)
    
    print(f"\n✅ 标准 EMD 分解预处理完成！数据已保存至: {output_dir}")

if __name__ == "__main__":
    main()

In [ ]:
# -*- coding: utf-8 -*-
"""
第二部分：标准EMD-SE-CNN 二分类网络训练与预测
"""
import os
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import random
import numpy as np
import tensorflow as tf

random.seed(2024)
np.random.seed(2024)
tf.random.set_seed(2024)

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Conv1D, MaxPooling1D, Dropout, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling1D, Reshape, Multiply, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

class SEBlock(tf.keras.layers.Layer):
    def __init__(self, reduction=3, **kwargs):
        super(SEBlock, self).__init__(**kwargs)
        self.reduction = reduction
        
    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.fc1 = Dense(max(self.channels // self.reduction, 4), activation='relu', use_bias=False, kernel_initializer='he_normal')
        self.fc2 = Dense(self.channels, activation='sigmoid', use_bias=False, kernel_initializer='he_normal')
        super(SEBlock, self).build(input_shape)
        
    def call(self, inputs):
        se = GlobalAveragePooling1D()(inputs)
        se = Reshape((1, self.channels))(se)
        se = self.fc1(se)
        se = self.fc2(se)
        return Multiply()([inputs, se])
    
    def get_config(self):
        config = super(SEBlock, self).get_config()
        config.update({'reduction': self.reduction})
        return config

def main():
    print("=== 第二阶段：标准 EMD-SE-CNN 二分类窃电检测 ===")
    
    data_dir = "standard_emd_data"
    try:
        X_train = np.load(os.path.join(data_dir, "X_train.npy"))
        y_train = np.load(os.path.join(data_dir, "y_train.npy"))
        X_test = np.load(os.path.join(data_dir, "X_test.npy"))
        y_test = np.load(os.path.join(data_dir, "y_test.npy"))
        
        X_train = X_train[..., np.newaxis] 
        X_test = X_test[..., np.newaxis]   
        print(f"✅ 数据加载成功！训练集: {X_train.shape}, 测试集: {X_test.shape}")
    except FileNotFoundError:
        print("❌ 找不到数据文件夹，请先运行第一部分预处理脚本！")
        return
    
    inputs = Input(shape=X_train.shape[1:]) 
    
    x = Conv1D(32, 5, strides=1, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(inputs)
    x = SEBlock(reduction=3)(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2, padding='same')(x)
    x = Dropout(0.3)(x)
    
    x = Conv1D(64, 3, strides=1, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = SEBlock(reduction=3)(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2, padding='same')(x)
    x = Dropout(0.3)(x)
    
    x = Flatten()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(2e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    
    x = Dense(64, activation='relu', kernel_regularizer=l2(2e-4))(x)
    x = Dropout(0.4)(x)
    
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        loss='binary_crossentropy',
        optimizer=Adam(learning_rate=0.0005),
        metrics=['accuracy']
    )
    
    early_stop = EarlyStopping(monitor='val_accuracy', patience=25, restore_best_weights=True, verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)
    
    print("\n🚀 开始模型训练...")
    history = model.fit(
        X_train, y_train,
        epochs=150,
        batch_size=32,
        validation_data=(X_test, y_test), 
        callbacks=[early_stop, reduce_lr],
        shuffle=True,
        verbose=1
    )
    
    print("\n=== 模型硬核指标评估 ===")
    y_pred_proba = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    fpr_curve, tpr_curve, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr_curve, tpr_curve)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr_val = fp / (fp + tn + 1e-10)
    fnr_val = fn / (fn + tp + 1e-10)
    g_mean = np.sqrt((tp / (tp + fn + 1e-10)) * (tn / (tn + fp + 1e-10)))
    
    print("\n" + "="*50)
    print("=== SCI 级二分类详细性能指标 (标准EMD) ===")
    print("="*50)
    print(f"准确率 (Accuracy)       : {accuracy:.4f}")
    print(f"精确率 (Precision)      : {precision:.4f}")
    print(f"召回率 (Recall)         : {recall:.4f}")
    print(f"F1分数 (F1-Score)       : {f1:.4f}")
    print(f"AUC面积 (ROC-AUC)       : {roc_auc:.4f}")
    print("-" * 50)
    print(f"误报率 (FPR)            : {fpr_val:.4f}")
    print(f"漏报率 (FNR)            : {fnr_val:.4f}")
    print(f"综合均值 (G-Mean)       : {g_mean:.4f}")
    print(f"马修斯系数 (MCC)        : {mcc:.4f}")
    print("="*50)
    
    print("\n分类报告:")
    print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Theft (1)'], digits=4))
    
    save_path = "Binary_Results_CSV_Standard_EMD2"
    os.makedirs(save_path, exist_ok=True)
    df_save = pd.DataFrame({'label': y_test, 'pred': y_pred_proba})
    csv_filename = os.path.join(save_path, "Binary_Standard_EMD_CNN_SE2.csv")
    df_save.to_csv(csv_filename, index=False)
    print(f"\n✅ 预测结果已成功导出至: {csv_filename}")

if __name__ == "__main__":
    main()

In [ ]:
# -*- coding: utf-8 -*-
"""
第一部分：固定阈值 EMD 数据预处理（全局统一阈值，不强制剔除IMF1）
"""
import os
os.environ['PYTHONHASHSEED'] = '42'
import random
import numpy as np
random.seed(42)
np.random.seed(42)

import pandas as pd
from PyEMD import EMD
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

class FixedThresholdEMDProcessor:
    def __init__(self, n_imfs=8):
        self.n_imfs = n_imfs
        self.emd = EMD()

    def _calc_correlation(self, orig, recon):
        min_len = min(len(orig), len(recon))
        if min_len < 2: return 0.0
        orig, recon = orig[:min_len], recon[:min_len]
        if np.std(orig) < 1e-6 or np.std(recon) < 1e-6: return 0.0
        return np.corrcoef(orig, recon)[0, 1]

    def _fixed_denoise(self, sample):
        orig_signal = sample.copy()
        try:
            imfs = self.emd(orig_signal, max_imf=self.n_imfs)
            residue = orig_signal - np.sum(imfs, axis=0)
            if len(imfs) < 2: return orig_signal
            
            total_energy = np.sum(orig_signal ** 2) + 1e-10
            energy_ratios = []
            correlations = []
            for imf in imfs:
                energy_ratios.append(np.sum(imf ** 2) / total_energy)
                if np.std(imf) < 1e-6 or np.std(orig_signal) < 1e-6:
                    correlations.append(0)
                else:
                    correlations.append(np.corrcoef(orig_signal, imf)[0, 1])
            energy_ratios = np.array(energy_ratios)
            correlations = np.array(correlations)
            
            # 👇 核心：全局统一固定阈值，不分标签，不强制剔除IMF1
            useful_mask = (energy_ratios > 0.02*0.8) | (np.abs(correlations) > 0.12*0.8)
            useful_imfs = imfs[useful_mask]
            
            # 保底：如果一个都没筛出来，就全保留
            if len(useful_imfs) == 0:
                useful_imfs = imfs
                
            reconstructed = np.sum(useful_imfs, axis=0) + residue
            
            # 统一保底机制（与动态EMD一致）
            if self._calc_correlation(orig_signal, reconstructed) < 0.85:
                useful_mask = (energy_ratios > 0.01) | (np.abs(correlations) > 0.06)
                useful_imfs = imfs[useful_mask]
                reconstructed = np.sum(useful_imfs, axis=0) + residue
                if self._calc_correlation(orig_signal, reconstructed) < 0.85:
                    reconstructed = np.sum(imfs, axis=0) + residue
            
            # 幅值校准
            if np.max(reconstructed) != 0:
                reconstructed = reconstructed * (np.max(orig_signal) / np.max(reconstructed))
            return reconstructed[:len(orig_signal)]
        except Exception:
            return orig_signal

    def process_data(self, X_data):
        print(f"=== 固定阈值 EMD 处理 {len(X_data)} 个样本 ===")
        processed = []
        for idx, sample in enumerate(X_data):
            if idx % 200 == 0: print(f"进度: {idx}/{len(X_data)}")
            processed.append(self._fixed_denoise(sample))
        return np.array(processed)

def main():
    file_path = r'E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\xin\Electricity-Theft-Detection--master\sample.xlsx'
    try:
        df = pd.read_excel(file_path)
    except Exception:
        df = pd.read_csv(file_path, encoding='utf-8')
    if 'Unnamed: 0' in df.columns:
        df.drop(['Unnamed: 0'], axis=1, inplace=True)
    Y = df['output'].values
    X = df.drop(['output'], axis=1).values

    # ✅ 与动态EMD完全对齐：种子42，test_size=0.2
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, Y, test_size=0.2, random_state=42, stratify=Y
    )

    processor = FixedThresholdEMDProcessor(n_imfs=8)
    X_train = processor.process_data(X_train_raw)
    X_test = processor.process_data(X_test_raw)

    output_dir = "fixed_threshold_emd_data"
    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, "X_train.npy"), X_train)
    np.save(os.path.join(output_dir, "y_train.npy"), y_train_raw)
    np.save(os.path.join(output_dir, "X_test.npy"), X_test)
    np.save(os.path.join(output_dir, "y_test.npy"), y_test_raw)
    print(f"\n✅ 固定阈值 EMD 预处理完成！数据已保存至: {output_dir}")

if __name__ == "__main__":
    main()

In [ ]:
# -*- coding: utf-8 -*-
"""
第二部分：固定阈值 EMD-SE-CNN 二分类网络训练与预测
"""
import os
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import random
import numpy as np
import tensorflow as tf
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Conv1D, MaxPooling1D, Dropout, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling1D, Reshape, Multiply, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

class SEBlock(tf.keras.layers.Layer):
    def __init__(self, reduction=3, **kwargs):
        super(SEBlock, self).__init__(**kwargs)
        self.reduction = reduction
    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.fc1 = Dense(max(self.channels // self.reduction, 4), activation='relu', use_bias=False, kernel_initializer='he_normal')
        self.fc2 = Dense(self.channels, activation='sigmoid', use_bias=False, kernel_initializer='he_normal')
        super(SEBlock, self).build(input_shape)
    def call(self, inputs):
        se = GlobalAveragePooling1D()(inputs)
        se = Reshape((1, self.channels))(se)
        se = self.fc1(se)
        se = self.fc2(se)
        return Multiply()([inputs, se])
    def get_config(self):
        config = super(SEBlock, self).get_config()
        config.update({'reduction': self.reduction})
        return config

def main():
    print("=== 第二阶段：固定阈值 EMD-SE-CNN 二分类窃电检测 ===")
    data_dir = "fixed_threshold_emd_data"
    try:
        X_train = np.load(os.path.join(data_dir, "X_train.npy"))
        y_train = np.load(os.path.join(data_dir, "y_train.npy"))
        X_test = np.load(os.path.join(data_dir, "X_test.npy"))
        y_test = np.load(os.path.join(data_dir, "y_test.npy"))
        X_train = X_train[..., np.newaxis] 
        X_test = X_test[..., np.newaxis]   
        print(f"✅ 数据加载成功！训练集: {X_train.shape}, 测试集: {X_test.shape}")
    except FileNotFoundError:
        print("❌ 找不到数据文件夹，请先运行第一部分预处理脚本！")
        return

    inputs = Input(shape=X_train.shape[1:])
    x = Conv1D(32, 5, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(inputs)
    x = SEBlock(2)(x); x = BatchNormalization()(x); x = MaxPooling1D(2, padding='same')(x); x = Dropout(0.3)(x)
    x = Conv1D(64, 3, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = SEBlock(4)(x); x = BatchNormalization()(x); x = MaxPooling1D(2, padding='same')(x); x = Dropout(0.3)(x)
    x = Flatten()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(2e-4))(x)
    x = BatchNormalization()(x); x = Dropout(0.5)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(2e-4))(x); x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='binary_crossentropy', optimizer=Adam(0.0005), metrics=['accuracy'])

    early_stop = EarlyStopping(monitor='val_accuracy', patience=25, restore_best_weights=True, verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)

    print("\n🚀 开始模型训练...")
    history = model.fit(
        X_train, y_train,
        epochs=150, batch_size=32,
        validation_data=(X_test, y_test),
        callbacks=[early_stop, reduce_lr],
        shuffle=True, verbose=1
    )

    print("\n=== 模型硬核指标评估 ===")
    y_pred_proba = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    fpr_curve, tpr_curve, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr_curve, tpr_curve)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr_val = fp / (fp + tn + 1e-10)
    fnr_val = fn / (fn + tp + 1e-10)
    g_mean = np.sqrt((tp / (tp + fn + 1e-10)) * (tn / (tn + fp + 1e-10)))

    print("\n" + "="*50)
    print("=== SCI 级二分类详细性能指标 (固定阈值EMD) ===")
    print("="*50)
    print(f"准确率 (Accuracy)       : {accuracy:.4f}")
    print(f"精确率 (Precision)      : {precision:.4f}")
    print(f"召回率 (Recall)         : {recall:.4f}")
    print(f"F1分数 (F1-Score)       : {f1:.4f}")
    print(f"AUC面积 (ROC-AUC)       : {roc_auc:.4f}")
    print("-" * 50)
    print(f"误报率 (FPR)            : {fpr_val:.4f}")
    print(f"漏报率 (FNR)            : {fnr_val:.4f}")
    print(f"综合均值 (G-Mean)       : {g_mean:.4f}")
    print(f"马修斯系数 (MCC)        : {mcc:.4f}")
    print("="*50)

    save_path = "Binary_Results_CSV_Fixed_EMD"
    os.makedirs(save_path, exist_ok=True)
    df_save = pd.DataFrame({'label': y_test, 'pred': y_pred_proba})
    csv_filename = os.path.join(save_path, "Binary_Fixed_EMD_CNN_SE.csv")
    df_save.to_csv(csv_filename, index=False)
    print(f"\n✅ 预测结果已成功导出至: {csv_filename}")

if __name__ == "__main__":
    main()